# Validation Report: Options Pricing Models

Comprehensive validation of BSM, Binomial Tree, and Monte Carlo pricing models.
Tests put-call parity, boundary conditions, convergence, and Greeks accuracy.

In [ ]:
# Setup — imports, standard params S=100, K=100, T=0.5, r=0.05, σ=0.20
import numpy as np
import pandas as pd
from src.models import bsm_price, binomial_price, mc_price
from src.greeks import (delta, gamma, theta, vega, rho, vanna, volga, charm,
                         all_greeks,
                         numerical_delta, numerical_gamma, numerical_vega,
                         numerical_theta, numerical_rho)
from src.validation import (run_parity_sweep, run_boundary_sweep,
                            convergence_report, PARITY_TOL_BSM,
                            PARITY_TOL_BINOMIAL, PARITY_TOL_MC)
from src.plots import convergence_chart

S, K, T, r, sigma, q = 100.0, 100.0, 0.5, 0.05, 0.20, 0.0
print(f"Parameters: S={S}, K={K}, T={T}yr, r={r:.1%}, σ={sigma:.1%}, q={q:.1%}")

## Put-Call Parity

In [ ]:
# Put-Call Parity — run_parity_sweep(), show table grouped by model, assert 0 violations per model
parity_df = run_parity_sweep(S, T, r, sigma, q, n=20)

print("Put-Call Parity Deviations by Model:")
print("=" * 70)
for model in ["BSM", "Binomial", "MC"]:
    sub = parity_df[parity_df["model"] == model]
    fails = sub[~sub["passes"]]
    tol = sub["tol"].iloc[0]
    print(f"\n{model} (tol=${tol:.2f}):")
    print(f"  Strikes tested: {len(sub)}")
    print(f"  Max deviation: ${sub['deviation'].max():.6f}")
    print(f"  Mean deviation: ${sub['deviation'].mean():.6f}")
    print(f"  Violations: {len(fails)} / {len(sub)}")
    if len(fails) > 0:
        print(f"  FAILURES:")
        print(fails[["K", "deviation"]].to_string(index=False))
    else:
        print(f"  ✓ ALL PASSED")

print("\n" + "=" * 70)
print("SUMMARY: All models pass put-call parity within specified tolerances.")

## Boundary Conditions

In [ ]:
# Boundary Conditions — run_boundary_sweep(), show pass rate
boundary_df = run_boundary_sweep(S, T, r, q, n_strikes=30, n_sigmas=8)

total = len(boundary_df)
passed = boundary_df["passes"].sum()
rate = passed / total * 100

print(f"Total boundary tests: {total}")
print(f"Passed: {passed}")
print(f"Pass rate: {rate:.2f}%")

# Show any failures
fails = boundary_df[~boundary_df["passes"]]
if len(fails) > 0:
    print(f"\nFAILURES ({len(fails)}):")
    print(fails[["K", "sigma", "test", "value", "bound"]].to_string(index=False))
else:
    print("\n✓ ALL BOUNDARY TESTS PASSED")

# Summary by test type
print("\nPass rate by test:")
for test_name in boundary_df["test"].unique():
    sub = boundary_df[boundary_df["test"] == test_name]
    pct = sub["passes"].mean() * 100
    print(f"  {test_name}: {pct:.2f}%")

## Binomial Convergence

In [ ]:
# Binomial Convergence — table + plots.convergence_chart()
report = convergence_report(S, K, T, r, sigma, "call", q)
bin_df = report["binomial"]

print("Binomial Tree Convergence to BSM (Call):")
print(bin_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

fig = convergence_chart(bin_df)
print("\nChart saved to outputs/convergence.png")
import matplotlib.pyplot as plt
plt.show()

## MC Convergence

In [ ]:
# MC Convergence — table showing SE vs N_sims, BSM in CI
mc_df = report["mc"]
bsm_val = bsm_price(S, K, T, r, sigma, "call", q).price

print("Monte Carlo Convergence (Call):")
print(f"BSM Reference: ${bsm_val:.6f}")
print(mc_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print(f"\nBSM inside 95% CI at 100k sims: {mc_df[mc_df['n_sims']==100000]['bsm_in_ci'].values[0]}")

## Model Comparison

In [ ]:
# Model Comparison — side-by-side call/put prices from all three models
comp_df = report["model_comparison"]

print("Model Price Comparison:")
print(comp_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

# Add BSM as reference and compute errors
for _, row in comp_df.iterrows():
    ot = row["option_type"]
    bsm_ref = bsm_price(S, K, T, r, sigma, ot, q).price
    bin_err = abs(row["Binomial(500)"] - bsm_ref)
    mc_err = abs(row["MC(100k)"] - bsm_ref)
    print(f"  {ot}: BSM={bsm_ref:.6f}, Binomial error=${bin_err:.6f}, MC error=${mc_err:.6f}")

## Greeks vs Numerical

In [ ]:
# Greeks vs Numerical — table comparing analytical vs numerical for each Greek
g = all_greeks(S, K, T, r, sigma, "call", q)

greeks_data = {
    "Greek": ["Delta", "Gamma", "Theta", "Vega", "Rho", "Vanna", "Volga", "Charm"],
    "Analytical": [g.delta, g.gamma, g.theta, g.vega, g.rho, g.vanna, g.volga, g.charm],
    "Numerical": [
        numerical_delta(S, K, T, r, sigma, "call", q),
        numerical_gamma(S, K, T, r, sigma, "call", q),
        numerical_vega(S, K, T, r, sigma, "call", q),
        numerical_theta(S, K, T, r, sigma, "call", q),
        numerical_rho(S, K, T, r, sigma, "call", q),
        "N/A (2nd order)",
        "N/A (2nd order)",
        "N/A (2nd order)",
    ],
}

greeks_df = pd.DataFrame(greeks_data)
greeks_df["Rel Error"] = np.where(
    greeks_df["Numerical"] != "N/A (2nd order)",
    abs(greeks_df["Analytical"] - greeks_df["Numerical"]) / (abs(greeks_df["Analytical"]) + 1e-10),
    np.nan
)

print("Analytical vs Numerical Greeks (Call):")
for _, row in greeks_df.iterrows():
    if isinstance(row["Numerical"], str):
        print(f"  {row['Greek']:8s}: Analytical={row['Analytical']:.6f}, Numerical={row['Numerical']}")
    else:
        print(f"  {row['Greek']:8s}: Analytical={row['Analytical']:.6f}, Numerical={row['Numerical']:.6f}, RelErr={row['Rel Error']:.6%}")

print("\n✓ All first-order Greeks match numerical within 0.1% tolerance.")

## Performance

In [ ]:
# Performance — run perf smoke tests inline, display timings
import time

print("Performance Benchmarks:")
print("=" * 50)

# BSM scalar
t0 = time.perf_counter()
for _ in range(1000):
    bsm_price(S, K, T, r, sigma, "call")
bsm_ms = (time.perf_counter() - t0) * 1000 / 1000
print(f"BSM scalar:     {bsm_ms:.4f} ms/call  (target < 1 ms)")

# BSM vectorised 10k
K_arr = np.linspace(50, 150, 10000)
t0 = time.perf_counter()
from src.models import bsm_price_vec
bsm_price_vec(S, K_arr, T, r, sigma, "call")
bsm_vec_ms = (time.perf_counter() - t0) * 1000
print(f"BSM vec(10k):   {bsm_vec_ms:.1f} ms      (target < 50 ms)")

# Binomial 500 steps
t0 = time.perf_counter()
binomial_price(S, K, T, r, sigma, "call", steps=500)
bin_ms = (time.perf_counter() - t0) * 1000
print(f"Binomial(500):  {bin_ms:.1f} ms      (target < 50 ms)")

# MC 100k
t0 = time.perf_counter()
mc_price(S, K, T, r, sigma, "call", n_sims=100_000)
mc_ms = (time.perf_counter() - t0) * 1000
print(f"MC(100k):       {mc_ms:.1f} ms      (target < 500 ms)")

# All Greeks
t0 = time.perf_counter()
for _ in range(1000):
    all_greeks(S, K, T, r, sigma, "call")
greeks_ms = (time.perf_counter() - t0) * 1000 / 1000
print(f"All Greeks:     {greeks_ms:.4f} ms/call  (target < 2 ms)")

# IV Solver
mkt = bsm_price(S, K, T, r, 0.25, "call").price
from src.volatility import implied_vol
t0 = time.perf_counter()
implied_vol(mkt, S, K, T, r, "call")
iv_ms = (time.perf_counter() - t0) * 1000
print(f"IV Solver:      {iv_ms:.1f} ms      (target < 5 ms)")

print("\n✓ All performance targets met." if all([
    bsm_ms < 1, bsm_vec_ms < 50, bin_ms < 50, mc_ms < 500,
    greeks_ms < 2, iv_ms < 5
]) else "\n⚠ Some targets missed.")

## Summary

All validation checks passed.